[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C13_RL_Foundations_Course/04_actor_critic_gae/04_actor_critic_gae.ipynb)

# 04 · Actor-Critic 与 GAE（纯 numpy）

目标：从零实现 **actor-critic**、**advantage**、**GAE(λ)**、**A2C**，在 toy chain 上收敛，并**数值验证** GAE 的三大性质：递归==显式和、λ=0→TD、λ=1→MC、方差随 λ 单调上升。

路线：TD 误差=advantage 估计 → GAE 递归(对拍显式和) → λ 两端退化 → 方差随 λ 上升(~100×) → A2C 完整环路收敛 → ✏️ 练习 → 📖 答案 → 🧪 PPO 风味胶囊。

> 心智模型：**critic 打分(V)，actor 据 advantage 更新**。GAE 用 λ 在「单步 TD(低方差高偏差)」与「蒙特卡洛(高方差无偏)」间连续滑动。你拼出的 A2C 就是 PPO 的骨架。

## 1 · TD 误差 = advantage 的无偏估计

actor-critic 的支柱：当 $V=V^\pi$ 时，$\mathbb{E}[\delta_t|s_t,a_t] = A^\pi(s_t,a_t)$。

先把 TD 误差和环境/策略工具搭好。用 chain MDP（同模块 03）。

In [ ]:
import numpy as np

def softmax(logits):
    z = logits - logits.max(); e = np.exp(z); return e / e.sum()

def score(theta_s, a):
    pi = softmax(theta_s); g = -pi.copy(); g[a] += 1.0; return g

class Chain:
    def __init__(self, N=6, gamma=0.99, step_reward=-0.05):
        self.N = N; self.gamma = gamma; self.step_reward = step_reward
        self.start = 0; self.goal = N - 1
    def reset(self): self.pos = self.start; return self.pos
    def step(self, a):
        self.pos = min(self.pos + 1, self.N - 1) if a == 1 else max(self.pos - 1, 0)
        done = (self.pos == self.goal)
        return self.pos, (1.0 if done else self.step_reward), done

def rollout(env, theta, rng, max_steps=50):
    s = env.reset(); S, A, Rw = [], [], []
    for _ in range(max_steps):
        a = int(rng.choice(2, p=softmax(theta[s])))
        sp, r, done = env.step(a)
        S.append(s); A.append(a); Rw.append(r); s = sp
        if done: break
    return S, A, Rw, done

def td_errors(rewards, values, gamma, last_value=0.0):
    '''δ_t = r_t + γ V(s_{t+1}) − V(s_t)。values[t]=V(s_t)，last_value=V(s_T)。'''
    T = len(rewards)
    deltas = np.zeros(T)
    for t in range(T):
        v_next = values[t + 1] if t + 1 < T else last_value
        deltas[t] = rewards[t] + gamma * v_next - values[t]
    return deltas

env = Chain(N=6, gamma=0.99)
rng = np.random.default_rng(0)
theta = np.zeros((env.N, 2))
S, A, Rw, done = rollout(env, theta, rng)
values = np.zeros(len(S))            # 暂用零 critic
deltas = td_errors(np.array(Rw), values, env.gamma, last_value=0.0)
print('一条轨迹的 TD 误差 δ:', np.round(deltas, 3))
assert len(deltas) == len(S)
# 零 critic 时 δ_t = r_t（末步含 +1）
assert np.allclose(deltas, Rw), '零 critic 时 δ 应等于即时奖励'
print('✅ TD 误差计算正确（零 critic 时 δ=r）')

## 2 · GAE：递归实现 对拍 显式几何和

GAE 定义 $\hat A_t = \sum_{l\ge0}(\gamma\lambda)^l \delta_{t+l}$，递归实现 $\hat A_t = \delta_t + \gamma\lambda\hat A_{t+1}$（$O(T)$ 反向扫描）。

两者必须逐位相等。这是 PPO 每次更新都在跑的核心子程序。

In [ ]:
def compute_gae(rewards, values, gamma, lam, last_value=0.0):
    '''GAE 递归：从后往前一遍扫描。'''
    T = len(rewards)
    adv = np.zeros(T)
    gae = 0.0
    for t in reversed(range(T)):
        v_next = values[t + 1] if t + 1 < T else last_value
        delta = rewards[t] + gamma * v_next - values[t]
        gae = delta + gamma * lam * gae        # Â_t = δ_t + γλ Â_{t+1}
        adv[t] = gae
    return adv

def gae_explicit(rewards, values, gamma, lam, last_value=0.0):
    '''按定义显式求和（慢，仅用于对拍）。'''
    deltas = td_errors(rewards, values, gamma, last_value)
    T = len(rewards); adv = np.zeros(T)
    for t in range(T):
        adv[t] = sum((gamma * lam) ** l * deltas[t + l] for l in range(T - t))
    return adv

rng = np.random.default_rng(1)
rewards = rng.standard_normal(7)
values = rng.standard_normal(7)
for lam in [0.0, 0.5, 0.7, 0.95, 1.0]:
    a_rec = compute_gae(rewards, values, 0.99, lam, last_value=0.0)
    a_exp = gae_explicit(rewards, values, 0.99, lam, last_value=0.0)
    assert np.allclose(a_rec, a_exp, atol=1e-12), f'λ={lam} 递归应等于显式和'
print('GAE 递归 == 显式几何和（所有 λ，机器精度）')
print('✅ GAE 递归实现正确（O(T) 反向扫描 = O(T²) 显式求和）')

## 3 · λ 两端退化：λ=0→TD，λ=1→MC

GAE 的两个极端必须分别退化为单步 TD 误差和蒙特卡洛 advantage。这是理解 λ 旋钮的关键验证。

In [ ]:
rng = np.random.default_rng(2)
rewards = rng.standard_normal(8)
values = rng.standard_normal(8)
gamma = 0.99

# λ=0 应等于单步 TD 误差 δ_t
adv_lam0 = compute_gae(rewards, values, gamma, 0.0, last_value=0.0)
deltas = td_errors(rewards, values, gamma, last_value=0.0)
assert np.allclose(adv_lam0, deltas, atol=1e-12), 'λ=0 应退化为单步 TD'
print('λ=0 GAE     :', np.round(adv_lam0, 3))
print('单步 TD 误差 :', np.round(deltas, 3), ' -> 相等 ✅')

# λ=1 应等于蒙特卡洛 advantage  G_t − V(s_t)
adv_lam1 = compute_gae(rewards, values, gamma, 1.0, last_value=0.0)
G = np.zeros(len(rewards)); run = 0.0
for t in reversed(range(len(rewards))):
    run = rewards[t] + gamma * run; G[t] = run
mc_adv = G - values
assert np.allclose(adv_lam1, mc_adv, atol=1e-10), 'λ=1 应退化为蒙特卡洛 advantage'
print('\nλ=1 GAE       :', np.round(adv_lam1, 3))
print('MC adv (G−V)  :', np.round(mc_adv, 3), ' -> 相等 ✅')
print('\n✅ λ=0→单步TD(低方差高偏差)，λ=1→蒙特卡洛(高方差无偏) —— GAE 是两者的插值')

## 4 · bias-variance 铁证：方差随 λ 单调上升

在固定策略 + **准确 critic**(真 V)下，advantage 的真值约为 0(动作无优劣时)，但其**估计的方差**应随 λ 增大。

用随机 chain + 蒙特卡洛估出的真 V，采大量 GAE advantage，验证方差随 λ 单调上升。

In [ ]:
class StochChain:
    def __init__(self, N=8, gamma=0.99, step_reward=-0.02, slip=0.15):
        self.N = N; self.gamma = gamma; self.step_reward = step_reward
        self.slip = slip; self.start = 0; self.goal = N - 1
    def reset(self): self.pos = self.start; return self.pos
    def step(self, a, rng):
        if rng.random() < self.slip: a = 1 - a       # 随机滑向反向
        self.pos = min(self.pos + 1, self.N - 1) if a == 1 else max(self.pos - 1, 0)
        done = (self.pos == self.goal)
        return self.pos, (1.0 if done else self.step_reward), done

def rollout_stoch(env, theta, rng, max_steps=100):
    s = env.reset(); S, A, Rw = [], [], []
    for _ in range(max_steps):
        a = int(rng.choice(2, p=softmax(theta[s])))
        sp, r, done = env.step(a, rng)
        S.append(s); A.append(a); Rw.append(r); s = sp
        if done: break
    return S, A, Rw, done

envs = StochChain(N=8)
theta_fix = np.zeros((envs.N, 2)); theta_fix[:, 1] = np.log(0.8 / 0.2)  # ~0.8 向右
# 蒙特卡洛估真 V
def mc_V(env, theta, rng, n=20000):
    sums = np.zeros(env.N); cnt = np.zeros(env.N)
    for _ in range(n):
        S, A, Rw, done = rollout_stoch(env, theta, rng)
        G = np.zeros(len(Rw)); run = 0.0
        for t in reversed(range(len(Rw))): run = Rw[t] + env.gamma * run; G[t] = run
        for i, st in enumerate(S): sums[st] += G[i]; cnt[st] += 1
    return np.where(cnt > 0, sums / np.maximum(cnt, 1), 0.0)
V_true = mc_V(envs, theta_fix, np.random.default_rng(1))

def adv_variance(env, theta, V, lam, n=4000, seed=3):
    rng = np.random.default_rng(seed); a0 = []
    for _ in range(n):
        S, A, Rw, done = rollout_stoch(env, theta, rng)
        values = np.array([V[st] for st in S])
        last_v = 0.0 if done else V[S[-1]]
        adv = compute_gae(np.array(Rw), values, env.gamma, lam, last_v)
        a0.append(adv[0])
    return np.mean(a0), np.var(a0)

print(f"{'λ':>5} {'mean(adv)':>12} {'var(adv)':>12}")
variances = []
for lam in [0.0, 0.5, 0.9, 0.95, 1.0]:
    m, v = adv_variance(envs, theta_fix, V_true, lam)
    variances.append(v)
    print(f'{lam:>5.2f} {m:>+12.4f} {v:>12.4f}')
# 方差应随 λ 单调上升
assert all(variances[i] <= variances[i + 1] + 1e-9 for i in range(len(variances) - 1)), \
    '方差应随 λ 单调上升'
assert variances[-1] > 10 * variances[0], 'λ=1 方差应远大于 λ=0(数量级差)'
print(f'\n方差 λ=0→λ=1 放大 {variances[-1]/variances[0]:.0f}× | 均值(真 advantage≈0)基本不变')
print('✅ bias-variance 铁证：λ 越大方差越大(准 critic 下均值不变) —— 这就是 λ 旋钮')

## 5 · A2C：actor + critic + GAE 完整环路

把三者组装：critic 算 δ、GAE 聚成 advantage、actor 据 advantage 更新、critic 回归到 TD(λ) 回报。

**关键**：critic 目标 $R_t = \hat A_t + V(s_t)$ 当常数(stop-gradient)；critic 学习率通常大于 actor。

In [ ]:
def a2c(env, n_episodes=3000, lr_actor=0.2, lr_critic=0.3, lam=0.95,
        seed=0, max_steps=50, entropy_beta=0.0):
    rng = np.random.default_rng(seed)
    theta = np.zeros((env.N, 2))    # actor
    V = np.zeros(env.N)             # critic (表格)
    returns = []
    for ep in range(n_episodes):
        S, A, Rw, done = rollout(env, theta, rng, max_steps)
        returns.append(sum(Rw))
        values = np.array([V[st] for st in S])
        last_v = 0.0 if done else V[S[-1]]     # 终止则自举值=0，截断则用 V
        adv = compute_gae(np.array(Rw), values, env.gamma, lam, last_v)
        v_target = adv + values               # critic 回归目标(TD(λ)回报)，视为常数
        # 更新 actor（advantage 当常数）
        for t, (st, at) in enumerate(zip(S, A)):
            g = adv[t] * score(theta[st], at)
            if entropy_beta > 0:               # 可选熵正则
                pi = softmax(theta[st]); H_grad = -pi * (np.log(pi + 1e-12) - np.log(pi + 1e-12) @ pi)
                g = g + entropy_beta * (-pi * (np.log(pi + 1e-12) + (-(pi * np.log(pi + 1e-12)).sum())))
            theta[st] += lr_actor * g
        # 更新 critic（回归到 v_target）
        for t, st in enumerate(S):
            V[st] += lr_critic * (v_target[t] - V[st])
    return theta, V, np.array(returns)

theta_a2c, V_a2c, ret_a2c = a2c(env, n_episodes=3000, lam=0.95, seed=0)
policy = [int(softmax(theta_a2c[s]).argmax()) for s in range(env.N)]
print(f'A2C 末期平均回报 = {ret_a2c[-200:].mean():.3f} (初期 {ret_a2c[:200].mean():.3f})')
print('学到策略(0=左,1=右):', policy[:-1])
print('critic V:', np.round(V_a2c, 3))
assert all(p == 1 for p in policy[:-1]), 'actor 应学会一路向右'
assert ret_a2c[-200:].mean() > ret_a2c[:200].mean(), '回报应上升'
# critic 应学到递增的 V（越靠近目标价值越高）
V_internal = V_a2c[:env.N - 1]
assert np.all(np.diff(V_internal) > -0.05), 'critic 的 V 应大致随接近目标递增'
print('✅ A2C 收敛：actor 学会向右，critic 学到递增价值，回报上升')

## 6 · A2C vs REINFORCE：更新信号的方差

A2C 用 GAE(λ=0.95)+critic 的 advantage 当更新信号；REINFORCE 用蒙特卡洛回报(等价 λ=1、且不减 baseline)。

直接比较喂给 actor 的**更新信号的方差**(这是 critic+GAE 价值的本质所在)：A2C 的信号方差应**明显更小**。

In [ ]:
# 固定一个策略 + 准 critic，比较两种更新信号(权重)的方差
envs2 = StochChain(N=8)
theta_cmp = np.zeros((envs2.N, 2)); theta_cmp[:, 1] = np.log(0.8 / 0.2)
V_acc = mc_V(envs2, theta_cmp, np.random.default_rng(7))    # 准 critic(复用第4节)

def signal_variance(env, theta, V, mode, n=4000, seed=8):
    '''采样每条轨迹起点的更新信号，返回其方差。
       mode=a2c: GAE(λ=0.95) advantage；mode=reinforce: 蒙特卡洛回报(无 baseline)。'''
    rng = np.random.default_rng(seed); sig = []
    for _ in range(n):
        S, A, Rw, done = rollout_stoch(env, theta, rng)
        if mode == 'a2c':
            values = np.array([V[st] for st in S]); last_v = 0.0 if done else V[S[-1]]
            w = compute_gae(np.array(Rw), values, env.gamma, 0.95, last_v)
        else:  # reinforce: 蒙特卡洛回报(无 baseline)
            G = np.zeros(len(Rw)); run = 0.0
            for t in reversed(range(len(Rw))): run = Rw[t] + env.gamma * run; G[t] = run
            w = G
        sig.append(w[0])         # 起点的更新信号
    return np.var(sig)

var_a2c = signal_variance(envs2, theta_cmp, V_acc, 'a2c')
var_rf  = signal_variance(envs2, theta_cmp, V_acc, 'reinforce')
print(f'A2C(GAE λ=0.95) 更新信号方差   = {var_a2c:.4f}')
print(f'REINFORCE(MC 回报) 更新信号方差 = {var_rf:.4f}')
print(f'方差缩减 {var_rf / var_a2c:.0f}×')
assert var_a2c < var_rf, 'A2C 的 GAE+critic 信号应比 REINFORCE 的 MC 回报方差更小'
print('✅ A2C 的更新信号(GAE+critic advantage)方差远小于 REINFORCE 的 MC 回报 —— 这就是 critic 的价值')

---
## ✏️ 练习 1：实现 GAE（递归）

实现 `gae(deltas, gamma, lam)`：给定一条轨迹的 TD 误差序列 `deltas`，返回 GAE advantage。

递归：$\hat A_t = \delta_t + \gamma\lambda\hat A_{t+1}$，从后往前。

In [ ]:
def gae(deltas, gamma, lam):
    adv = np.zeros(len(deltas))
    # TODO: 从后往前，running = delta[t] + gamma*lam*running，存入 adv[t]
    raise NotImplementedError
    return adv

In [ ]:
# —— 练习 1 自测 ——
deltas = np.array([1.0, 0.5, -0.2, 0.3])
a = gae(deltas, gamma=0.9, lam=0.5)
# 手算最后一项 = 0.3；倒数第二 = -0.2 + 0.45*0.3 = -0.065 ...
assert abs(a[-1] - 0.3) < 1e-9, '末项 = δ_T'
assert abs(a[2] - (-0.2 + 0.45 * 0.3)) < 1e-9, '递归应为 δ_t + γλ·Â_{t+1}'
# λ=0 时应等于 deltas 本身
assert np.allclose(gae(deltas, 0.9, 0.0), deltas), 'λ=0 → Â=δ'
print('✅ 练习 1 通过：GAE 递归正确')

## ✏️ 练习 2：critic 回归目标

实现 `critic_targets(adv, values)`：critic 应回归到 TD(λ) 回报 $R_t = \hat A_t + V(s_t)$。

返回该目标数组（这是 critic 的监督信号，训练时当常数）。

In [ ]:
def critic_targets(adv, values):
    # TODO: 返回 adv + values（TD(λ) 回报）
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
adv = np.array([0.5, -0.3, 0.2])
values = np.array([1.0, 1.2, 0.8])
tgt = critic_targets(adv, values)
assert np.allclose(tgt, [1.5, 0.9, 1.0]), 'R_t = Â_t + V(s_t)'
# 性质：critic 朝目标更新会减小 |Â|（advantage 趋于被 critic 吸收）
V_new = values + 0.5 * (tgt - values)   # 半步更新
new_adv_approx = tgt - V_new            # 更新后的残差
assert np.all(np.abs(new_adv_approx) <= np.abs(adv) + 1e-9), 'critic 学习应缩小 advantage 残差'
print('✅ 练习 2 通过：critic 目标 = advantage + V（TD(λ) 回报）')

## ✏️ 练习 3：n-step return

GAE 是所有 n-step 回报的加权平均。实现单个 `n_step_return(rewards, values, gamma, n, t, last_value)`：
返回从 t 起的 n-step 回报 $\sum_{k=0}^{n-1}\gamma^k r_{t+k} + \gamma^n V(s_{t+n})$（注意 n 步后若超出轨迹用 last_value 或末态）。

In [ ]:
def n_step_return(rewards, values, gamma, n, t, last_value=0.0):
    T = len(rewards)
    # TODO: 累加 sum_{k=0}^{min(n,T-t)-1} gamma^k * rewards[t+k]
    #       再加自举项 gamma^steps * V(s_{t+steps})，其中 s_{t+steps} 越界则用 last_value
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
rewards = np.array([1.0, 2.0, 3.0, 4.0])
values = np.array([10.0, 20.0, 30.0, 40.0])
gamma = 0.5
# 1-step from t=0: r0 + γ V(s1) = 1 + 0.5*20 = 11
assert abs(n_step_return(rewards, values, gamma, 1, 0) - 11.0) < 1e-9
# 2-step from t=0: r0 + γ r1 + γ² V(s2) = 1 + 0.5*2 + 0.25*30 = 9.5
assert abs(n_step_return(rewards, values, gamma, 2, 0) - 9.5) < 1e-9
# n 超过剩余长度: 退化为蒙特卡洛(从 t 到末尾 + last_value 自举)
mc = n_step_return(rewards, values, gamma, 100, 0, last_value=0.0)
manual = 1 + 0.5*2 + 0.25*3 + 0.125*4    # 全部真实奖励，无自举(last_value=0)
assert abs(mc - manual) < 1e-9, 'n→∞ 应退化为蒙特卡洛回报'
print('✅ 练习 3 通过：n-step 回报正确（n=1 是 TD，n→∞ 是 MC）')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def gae(deltas, gamma, lam):
    adv = np.zeros(len(deltas)); running = 0.0
    for t in reversed(range(len(deltas))):
        running = deltas[t] + gamma * lam * running
        adv[t] = running
    return adv

In [ ]:
# 练习 2 参考答案
def critic_targets(adv, values):
    return adv + values

In [ ]:
# 练习 3 参考答案
def n_step_return(rewards, values, gamma, n, t, last_value=0.0):
    T = len(rewards)
    steps = min(n, T - t)
    ret = sum(gamma ** k * rewards[t + k] for k in range(steps))
    boot = values[t + steps] if t + steps < T else last_value
    ret += gamma ** steps * boot
    return ret

---
## 🧪 真实数据胶囊：PPO 风味的裁剪更新

PPO(C41/C22 的主力)= A2C(actor-critic+GAE) + **裁剪**限制每次策略更新幅度，防过早塌缩/发散。

核心是 PPO 的裁剪目标：用新旧策略的概率比 $r = \pi_{new}/\pi_{old}$，把目标裁到 $[1-\epsilon, 1+\epsilon]$，避免一次更新把策略推太远。用真实的 PPO 默认超参($\epsilon=0.2$)实现这个裁剪目标。

In [ ]:
def ppo_clip_objective(ratio, advantage, eps=0.2):
    '''PPO 裁剪目标(单样本)：min(r·A, clip(r,1-ε,1+ε)·A)。
       ratio = π_new(a|s)/π_old(a|s)；advantage = Â。'''
    unclipped = ratio * advantage
    clipped = np.clip(ratio, 1 - eps, 1 + eps) * advantage
    return np.minimum(unclipped, clipped)

eps = 0.2   # PPO 默认
# 正 advantage：想推高该动作概率，但裁剪限制了上涨幅度
obj_pos = ppo_clip_objective(np.array([1.0, 1.5, 3.0]), advantage=1.0, eps=eps)
print('正 advantage, ratio=[1.0,1.5,3.0]:', np.round(obj_pos, 3))
# ratio=1.5 和 3.0 都被裁到 1.2(=1+ε)，目标=1.2*1=1.2（防止过度更新）
assert abs(obj_pos[1] - 1.2) < 1e-9 and abs(obj_pos[2] - 1.2) < 1e-9, '正 adv 时上涨被裁到 1+ε'
assert abs(obj_pos[0] - 1.0) < 1e-9, 'ratio=1 时目标=advantage'
# 负 advantage：想压低该动作，裁剪限制下跌幅度
obj_neg = ppo_clip_objective(np.array([1.0, 0.5, 0.1]), advantage=-1.0, eps=eps)
print('负 advantage, ratio=[1.0,0.5,0.1]:', np.round(obj_neg, 3))
assert abs(obj_neg[1] - (-0.8)) < 1e-9, '负 adv 时下跌被裁到 1-ε'
print('✅ PPO 裁剪：限制每次更新幅度，是 A2C 走向稳定大规模训练的关键(RLHF 主力)')

**🧪 胶囊练习**：实现 `is_clipped(ratio, advantage, eps)`：判断某样本是否被裁剪(梯度因此为 0)。PPO 的精髓正是「更新太猛就不再给梯度」。返回布尔。提示：正 advantage 且 ratio>1+ε，或负 advantage 且 ratio<1−ε，则被裁。

In [ ]:
def is_clipped(ratio, advantage, eps=0.2):
    # TODO: 正 adv 且 ratio>1+eps -> True；负 adv 且 ratio<1-eps -> True；否则 False
    raise NotImplementedError

In [ ]:
# 自测
assert is_clipped(1.5, advantage=1.0, eps=0.2) == True, '正adv且涨太多->裁'
assert is_clipped(1.1, advantage=1.0, eps=0.2) == False, '正adv小涨->不裁'
assert is_clipped(0.5, advantage=-1.0, eps=0.2) == True, '负adv且跌太多->裁'
assert is_clipped(0.9, advantage=-1.0, eps=0.2) == False, '负adv小跌->不裁'
print('✅ 胶囊练习通过：PPO 在更新过猛时停止梯度(信赖域思想)')

In [ ]:
# 📖 胶囊参考答案
def is_clipped(ratio, advantage, eps=0.2):
    if advantage > 0:
        return ratio > 1 + eps
    else:
        return ratio < 1 - eps

### 小结
- **actor-critic** = 策略梯度(actor) + 价值估计(critic)：用 critic 提供低方差 advantage 替代 MC 回报。
- **TD 误差 δ = advantage 的无偏估计**(当 V=V^π)：只学 V(不用学 Q)就够。
- **GAE(λ)**：Â_t = Σ(γλ)^l δ_{t+l} = δ_t + γλ Â_{t+1}(O(T) 递归)；**λ=0→TD，λ=1→MC**。
- **bias-variance 旋钮**：λ 越大方差越大、越无偏；方差才是主敌，λ≈0.95 是甜点。
- **A2C** = actor + critic + GAE；**你拼出的 A2C 就是 PPO 的骨架**(PPO = A2C + 裁剪)。

下一站：**模块 05 · 多臂老虎机与探索** —— 把 explore-exploit 这个 RL 灵魂问题剥离到最纯形式讲透。